## Writing simple neural network from scratch - train, test on MNIST dataset

In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# MNIST dataset
mnist = fetch_openml('mnist_784', version=1)
X = mnist.data.astype(np.float32)
y = mnist.target.astype(int)

# normalising inputs
X /= 255.0

# train, test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)  # (56000, 784)

# ensuring correct datatypes
X_train = np.array(X_train, dtype=np.float32)
X_test  = np.array(X_test, dtype=np.float32)
y_train = np.array(y_train, dtype=int)
y_test  = np.array(y_test, dtype=int)

(56000, 784)


Softmax function: $ Softmax(x_i) = \frac{e^{x_i}}{\sum e^{x_j}} $.


In [2]:
# hidden layer(s) activation function
def relu(x):
    return np.maximum(0, x)

# final output layer activation function
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))  # we subtract from the max since it ensures stability (at no cost)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

Cross entropy loss function: $L = -\sum_{c=1}^{M} y_{o,c} \log(p_{o,c})$ 

where $ M $= total number of classes, 

$y_{o,c}$ is truth value (actual class) - one hot encoded, 

$p_{o,c}$ is the predicted probability that observation $o$ belongs to class $c$ (softmax function output)



In [3]:
# loss function
def cross_entropy_loss(y_pred, y_true):

    n = y_pred.shape[0]
    log_likelihood = -np.log(y_pred[range(n), y_true] + 1e-12) # add small value to prevent log(0), ensure numerical stability
    return np.mean(log_likelihood)

In the neural network, we will impliment one hidden layer - relu (and one output layer - softmax)

For computing backpropagation step, partial derivative of loss with respect to softmax input is $\frac{\partial L}{\partial z_2} = \text{Predictions} - \text{Targets}$



In [4]:
class NeuralNetwork:
    
    # pass in the input, hidden layer, output sizes, learning rate
    def __init__(self, input_size, hidden_size, output_size, lr=0.01):

        self.lr = lr
        
        # random initialisation of weights and biases with He initialization (for ReLU)
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2. / input_size)
        self.b1 = np.zeros((1, hidden_size))
        
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2. / hidden_size)
        self.b2 = np.zeros((1, output_size))


    def forward(self, X):

        # computing dot product between inputs and weights, then adding bias, then applying activation function
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = relu(self.z1)
        
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = softmax(self.z2)
        
        return self.a2

    def backward(self, X, y):

        n = X.shape[0]
        
        grad_z2 = self.a2.copy()
        grad_z2[np.arange(n), y] -= 1
        grad_z2 /= n
        
        # softmax layer gradients
        grad_W2 = np.dot(self.a1.T, grad_z2)
        grad_b2 = np.sum(grad_z2, axis=0, keepdims=True)
        
        grad_a1 = np.dot(grad_z2, self.W2.T)
        grad_z1 = grad_a1 * (self.z1 > 0)
        
        # relu layer gradients
        grad_W1 = np.dot(X.T, grad_z1)
        grad_b1 = np.sum(grad_z1, axis=0, keepdims=True)
        
        # subtracting lr from gradients to update weights and biases
        self.W2 -= self.lr * grad_W2
        self.b2 -= self.lr * grad_b2
        self.W1 -= self.lr * grad_W1
        self.b1 -= self.lr * grad_b1

    def train(self, X, y, epochs=10, batch_size=64):
        
        n = X.shape[0]
        
        for epoch in range(epochs):

            # shuffling order of data for each epoch
            indices = np.random.permutation(n)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            

            # stochastic gradient descent in batches
            for i in range(0, n, batch_size):

                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                
                self.forward(X_batch)
                self.backward(X_batch, y_batch)
            
            # quick loss check
            probs = self.forward(X[:1000])
            loss = cross_entropy_loss(probs, y[:1000])
            
            print(f"Epoch {epoch+1}, Loss: {loss:.4f}")


    def predict(self, X):
        
        probs = self.forward(X)
        return np.argmax(probs, axis=1) # returning the index of the highest probability as the predicted class

In [5]:
model = NeuralNetwork(input_size=784, hidden_size=128, output_size=10, lr=0.01)
model.train(X_train, y_train, epochs=15, batch_size=64)

Epoch 1, Loss: 0.4421
Epoch 2, Loss: 0.3186
Epoch 3, Loss: 0.2743
Epoch 4, Loss: 0.2468
Epoch 5, Loss: 0.2263
Epoch 6, Loss: 0.2124
Epoch 7, Loss: 0.1993
Epoch 8, Loss: 0.1842
Epoch 9, Loss: 0.1765
Epoch 10, Loss: 0.1670
Epoch 11, Loss: 0.1598
Epoch 12, Loss: 0.1502
Epoch 13, Loss: 0.1443
Epoch 14, Loss: 0.1410
Epoch 15, Loss: 0.1329


In [6]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.97      0.97      1343
           1       0.96      0.98      0.97      1600
           2       0.94      0.93      0.94      1380
           3       0.93      0.92      0.93      1433
           4       0.92      0.94      0.93      1295
           5       0.92      0.94      0.93      1273
           6       0.95      0.96      0.96      1396
           7       0.95      0.95      0.95      1503
           8       0.94      0.91      0.93      1357
           9       0.93      0.92      0.93      1420

    accuracy                           0.94     14000
   macro avg       0.94      0.94      0.94     14000
weighted avg       0.94      0.94      0.94     14000



We now make a neural network with multiple hidden layers

In [7]:
class DeepNeuralNetwork:
    
    # two hidden relu layers, softmax output layer
    def __init__(self, input_size, hidden1_size, hidden2_size, output_size, lr=0.01):

        self.lr = lr
        
        self.W1 = np.random.randn(input_size, hidden1_size) * np.sqrt(2. / input_size)
        self.b1 = np.zeros((1, hidden1_size))
        
        self.W2 = np.random.randn(hidden1_size, hidden2_size) * np.sqrt(2. / hidden1_size)
        self.b2 = np.zeros((1, hidden2_size))

        self.W3 = np.random.randn(hidden2_size, output_size) * np.sqrt(2. / hidden2_size)
        self.b3 = np.zeros((1, output_size))


    def forward(self, X):

        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = relu(self.z1)
    
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = relu(self.z2)
        
        self.z3 = np.dot(self.a2, self.W3) + self.b3
        self.a3 = softmax(self.z3)
        
        return self.a3 # returning final output


    def backward(self, X, y):
        n = X.shape[0]
        
        grad_z3 = self.a3.copy()
        grad_z3[np.arange(n), y] -= 1
        grad_z3 /= n
        
        grad_W3 = np.dot(self.a2.T, grad_z3)
        grad_b3 = np.sum(grad_z3, axis=0, keepdims=True)
        
        grad_a2 = np.dot(grad_z3, self.W3.T)
        grad_z2 = grad_a2 * (self.z2 > 0)
        
        grad_W2 = np.dot(self.a1.T, grad_z2)
        grad_b2 = np.sum(grad_z2, axis=0, keepdims=True)

        grad_a1 = np.dot(grad_z2, self.W2.T)
        grad_z1 = grad_a1 * (self.z1 > 0)
        
        grad_W1 = np.dot(X.T, grad_z1)
        grad_b1 = np.sum(grad_z1, axis=0, keepdims=True)
        

        # update weights and biases
        self.W3 -= self.lr * grad_W3
        self.b3 -= self.lr * grad_b3
        self.W2 -= self.lr * grad_W2
        self.b2 -= self.lr * grad_b2
        self.W1 -= self.lr * grad_W1
        self.b1 -= self.lr * grad_b1


    def train(self, X, y, epochs=10, batch_size=64):

        n = X.shape[0]

        for epoch in range(epochs):

            indices = np.random.permutation(n)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            for i in range(0, n, batch_size):

                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                
                self.forward(X_batch)
                self.backward(X_batch, y_batch)
            
            # loss
            probs = self.forward(X[:1000])
            loss = cross_entropy_loss(probs, y[:1000])
            print(f"Epoch {epoch+1}, Loss: {loss:.4f}")


    def predict(self, X):
        probs = self.forward(X)
        return np.argmax(probs, axis=1)

In [8]:
model2 = DeepNeuralNetwork(input_size=784, hidden1_size=256, hidden2_size=128, output_size=10, lr=0.01)
model2.train(X_train, y_train, epochs=15, batch_size=64)

Epoch 1, Loss: 0.3308
Epoch 2, Loss: 0.2353
Epoch 3, Loss: 0.1916
Epoch 4, Loss: 0.1713
Epoch 5, Loss: 0.1484
Epoch 6, Loss: 0.1369
Epoch 7, Loss: 0.1230
Epoch 8, Loss: 0.1097
Epoch 9, Loss: 0.1029
Epoch 10, Loss: 0.0980
Epoch 11, Loss: 0.0936
Epoch 12, Loss: 0.0841
Epoch 13, Loss: 0.0782
Epoch 14, Loss: 0.0768
Epoch 15, Loss: 0.0715


In [9]:
y_pred2 = model2.predict(X_test)
print(classification_report(y_test, y_pred2))

              precision    recall  f1-score   support

           0       0.98      0.97      0.98      1343
           1       0.97      0.98      0.98      1600
           2       0.96      0.95      0.96      1380
           3       0.97      0.94      0.95      1433
           4       0.95      0.96      0.96      1295
           5       0.95      0.96      0.96      1273
           6       0.97      0.98      0.97      1396
           7       0.96      0.97      0.97      1503
           8       0.95      0.94      0.95      1357
           9       0.94      0.95      0.95      1420

    accuracy                           0.96     14000
   macro avg       0.96      0.96      0.96     14000
weighted avg       0.96      0.96      0.96     14000



In [10]:
print("One relu layer: ", accuracy_score(y_test, y_pred))
print("Two relu layers: ", accuracy_score(y_test, y_pred2))


One relu layer:  0.9435
Two relu layers:  0.9616428571428571


Implimenting a cnn with pytorch as a comparison to our models (should perform much better)

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# convert data to pytorch tensors and reshape for cnn (batch, channels, height, width)
X_train_torch = torch.from_numpy(X_train).reshape(-1, 1, 28, 28)
X_test_torch = torch.from_numpy(X_test).reshape(-1, 1, 28, 28)
y_train_torch = torch.from_numpy(y_train).long()
y_test_torch = torch.from_numpy(y_test).long()

# data loaders
train_dataset = TensorDataset(X_train_torch, y_train_torch)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# defining cnn architecture
class ConvolutionalNeuralNetwork(nn.Module):
    def __init__(self):
        super(ConvolutionalNeuralNetwork, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x))) # (batch, 32, 14, 14)
        x = self.pool(self.relu(self.conv2(x))) # (batch, 64, 7, 7)
        x = x.view(x.size(0), -1) # flatten
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# initialise model, loss function, optimiser
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cnn_model = ConvolutionalNeuralNetwork().to(device)
criterion = nn.CrossEntropyLoss()
optimiser = optim.Adam(cnn_model.parameters(), lr=0.001)

cnn_model.train()

for epoch in range(15):

    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimiser.zero_grad()
        outputs = cnn_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimiser.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")



Epoch 1, Loss: 0.1765
Epoch 2, Loss: 0.0496
Epoch 3, Loss: 0.0348
Epoch 4, Loss: 0.0253
Epoch 5, Loss: 0.0189
Epoch 6, Loss: 0.0157
Epoch 7, Loss: 0.0120
Epoch 8, Loss: 0.0085
Epoch 9, Loss: 0.0089
Epoch 10, Loss: 0.0074
Epoch 11, Loss: 0.0054
Epoch 12, Loss: 0.0053
Epoch 13, Loss: 0.0061
Epoch 14, Loss: 0.0041
Epoch 15, Loss: 0.0042


In [13]:
# evaluate on test set
cnn_model.eval()
with torch.no_grad():
    X_test_device = X_test_torch.to(device)
    outputs = cnn_model(X_test_device)
    y_pred_cnn = torch.argmax(outputs, dim=1).cpu().numpy()

print(classification_report(y_test, y_pred_cnn))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1343
           1       1.00      0.99      1.00      1600
           2       0.99      0.99      0.99      1380
           3       1.00      0.99      0.99      1433
           4       0.99      1.00      0.99      1295
           5       0.99      0.99      0.99      1273
           6       0.99      0.99      0.99      1396
           7       0.99      0.99      0.99      1503
           8       0.99      0.98      0.99      1357
           9       0.98      0.99      0.98      1420

    accuracy                           0.99     14000
   macro avg       0.99      0.99      0.99     14000
weighted avg       0.99      0.99      0.99     14000



In [17]:
# compare all three models
print("\n" + "="*50)
print("Accuracy Comparison:")
print("="*50)
print(f"Simple Neural Network (1 hidden layer):  {accuracy_score(y_test, y_pred):.4f}")
print(f"Deep Neural Network (2 hidden layers):   {accuracy_score(y_test, y_pred2):.4f}")
print(f"Convolutional Neural Network (pytorch): {accuracy_score(y_test, y_pred_cnn):.4f}")
print("="*50)


Accuracy Comparison:
Simple Neural Network (1 hidden layer):  0.9435
Deep Neural Network (2 hidden layers):   0.9616
Convolutional Neural Network (pytorch): 0.9912
